# Maintenance Rehearsal (A) — Training (pilot)

Trains `t5-small` on the (input_ids, attention_mask, labels) produced by
`notebooks/04_rehearsal_maintenance_prep.ipynb`. This is a pilot (`-small`);
once validated, scale up to `t5-base`.

Metrics:
- **ROUGE-L** — reference metric (standard for extractive summarization)
- **novel n-gram ratio** (manipulation check) — for A, this should trend
  toward 0 as training progresses. Must be compared against the **source
  text** (i.e. the untokenized input — what `chunk.text` would be in
  `rehearse_maintenance`), not the label (the label is already a subset of
  the source, so comparing against it would trivially look "close to 0" and
  tell us nothing). That's why this isn't in `compute_metrics` — a separate
  callback (`NovelNGramCallback`) generates from a slice of the validation
  set and compares directly against the source.

In [1]:
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()
API_KEY_SET = bool(os.environ.get("NVIDIA_NIM_API_KEY"))

import datasets
import evaluate
import numpy as np
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

from src.pipeline.rehearsal import novel_ngram_ratio


project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data/model

Loads the tokenized dataset (train 3,000-ish / val 300-ish / test 300-ish,
produced by `04_rehearsal_maintenance_prep.ipynb`) and subsamples down to a
size that actually finishes in reasonable time. `PILOT_TRAIN_SIZE`/
`PILOT_VAL_SIZE`/`PILOT_TEST_SIZE` exist purely to validate the code path —
this is not a quality benchmark. Scale them up once the path is confirmed
working (and re-measure step timing on your hardware before committing to a
bigger run).

`val` drives model selection during training (`load_best_model_at_end`/
`metric_for_best_model="rougeL"` below) — `test` is never touched until §5,
after training is completely finished, so the reported test metric isn't
inflated by having been used to pick the checkpoint.

In [ ]:
MODEL_NAME = "t5-small"
DATA_DIR = Path("data/processed/rehearsal_maintenance")
OUTPUT_DIR = "experiments/rehearsal_maintenance_small"
PILOT_TRAIN_SIZE = 200
PILOT_VAL_SIZE = 20
PILOT_TEST_SIZE = 20

DATA_READY = all((DATA_DIR / split).exists() for split in ("train", "val", "test"))
if not DATA_READY:
    print(f"No tokenized data at {DATA_DIR} — run 04_rehearsal_maintenance_prep.ipynb first.")
else:
    full_train_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "train"))
    full_val_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "val"))
    full_test_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "test"))

    train_dataset = full_train_dataset.select(range(min(PILOT_TRAIN_SIZE, len(full_train_dataset))))
    val_dataset = full_val_dataset.select(range(min(PILOT_VAL_SIZE, len(full_val_dataset))))
    test_dataset = full_test_dataset.select(range(min(PILOT_TEST_SIZE, len(full_test_dataset))))
    print(f"Using {len(train_dataset)}/{len(full_train_dataset)} train rows, "
          f"{len(val_dataset)}/{len(full_val_dataset)} val rows, "
          f"{len(test_dataset)}/{len(full_test_dataset)} test rows")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    print(f"{MODEL_NAME} loaded, parameter count: {sum(p.numel() for p in model.parameters()):,}")


## 2. Metrics

`compute_metrics` computes ROUGE-L only (prediction vs. label, the standard
way). The novel n-gram ratio is computed separately in `NovelNGramCallback`
below (it needs to compare against the *source* input_ids, which
`compute_metrics`'s (prediction, label) pair alone doesn't give us).

In [3]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {"rougeL": result["rougeL"]}


def average_novel_ngram_ratio(
    model, tokenizer, eval_dataset, n: int = 3, sample_size: int = 5, max_new_tokens: int = 256
) -> float:
    """Generates from a slice of eval_dataset and returns the average novel
    n-gram ratio against the *source* (input), not the label — shared by
    NovelNGramCallback (during training) and the final test-set evaluation
    (§5) so both use identical logic."""
    examples = eval_dataset.select(range(min(sample_size, len(eval_dataset))))
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device

    ratios = []
    for example in examples:
        input_ids = torch.tensor([example["input_ids"]]).to(device)
        with torch.no_grad():
            output_ids = model.generate(input_ids=input_ids, max_new_tokens=max_new_tokens)
        generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        source_text = tokenizer.decode(example["input_ids"], skip_special_tokens=True)
        ratios.append(novel_ngram_ratio(generated_text, source_text, n=n))

    if was_training:
        model.train()
    return sum(ratios) / len(ratios) if ratios else 0.0


class NovelNGramCallback(TrainerCallback):
    """Logs the novel n-gram ratio (manipulation check) on a validation slice
    at every eval. For A, this should trend toward 0 as training progresses —
    compared against the *source* (input), not the label (the label is
    already a subset of the source, so comparing against it would trivially
    look low and tell us nothing).
    """

    def __init__(self, tokenizer, eval_dataset, n: int = 3, sample_size: int = 5, max_new_tokens: int = 256):
        self.tokenizer = tokenizer
        self.eval_dataset = eval_dataset
        self.n = n
        self.sample_size = sample_size
        self.max_new_tokens = max_new_tokens

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        avg_ratio = average_novel_ngram_ratio(
            model, self.tokenizer, self.eval_dataset, self.n, self.sample_size, self.max_new_tokens
        )
        print(f"[novel {self.n}-gram ratio] step={state.global_step}: {avg_ratio:.4f} ({self.sample_size} eval samples)")


## 3. Train

`Seq2SeqTrainer` + `Seq2SeqTrainingArguments`. `predict_with_generate=True` is
required (ROUGE-L needs actual generated text — without it, `compute_metrics`
only gets raw logits and can't compute ROUGE).

Checkpoints go to `experiments/rehearsal_maintenance_small/` (reusing the
existing `experiments/` layout).

In [4]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=256,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[NovelNGramCallback(tokenizer, val_dataset)],
)

trainer.train()

  0%|          | 0/50 [00:00<?, ?it/s]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
 20%|██        | 10/50 [00:13<00:47,  1.19s/it]

{'loss': 0.9081, 'grad_norm': 4.241733551025391, 'learning_rate': 4e-05, 'epoch': 0.2}


 40%|████      | 20/50 [00:25<00:37,  1.24s/it]

{'loss': 0.5701, 'grad_norm': 3.987316131591797, 'learning_rate': 3e-05, 'epoch': 0.4}


 60%|██████    | 30/50 [00:36<00:26,  1.31s/it]

...

{'eval_loss': 0.2751997113227844, 'eval_rougeL': 0.21718144492121835, 'eval_runtime': 22.9095, 'eval_samples_per_second': 0.873, 'eval_steps_per_second': 0.218, 'epoch': 1.0}


[novel 3-gram ratio] step=50: 0.1060 (5 eval samples)


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
100%|██████████| 50/50 [01:31<00:00,  1.83s/it]

{'train_runtime': 91.5604, 'train_samples_per_second': 2.184, 'train_steps_per_second': 0.546, 'train_loss': 0.6199324178695679, 'epoch': 1.0}


TrainOutput(global_step=50, training_loss=0.6199324178695679, metrics={'train_runtime': 91.5604, 'train_samples_per_second': 2.184, 'train_steps_per_second': 0.546, 'total_flos': 26750095589376.0, 'train_loss': 0.6199324178695679, 'epoch': 1.0})

## 4. Save final model

In [5]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")

Saved to: experiments/rehearsal_maintenance_small


## 5. Final evaluation — held-out test set

`test_dataset` was never used during training (not for gradient updates, not
for checkpoint selection) — this is the one point where it gets touched, and
the resulting ROUGE-L is what should actually be reported/compared across
runs. `eval_rougeL` from the training log above is a validation-set number
used to pick the best checkpoint; reporting that as the final result would
be circular (the checkpoint was chosen because it scored well there).

In [6]:
if not DATA_READY:
    print("No data — skipping final test evaluation.")
else:
    test_trainer = Seq2SeqTrainer(
        model=model,
        args=Seq2SeqTrainingArguments(
            output_dir=OUTPUT_DIR,
            per_device_eval_batch_size=4,
            predict_with_generate=True,
            generation_max_length=256,
            report_to="none",
        ),
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    test_metrics = test_trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
    print("Final test-set metrics:", test_metrics)

    test_novel_ratio = average_novel_ngram_ratio(model, tokenizer, test_dataset, sample_size=min(10, len(test_dataset)))
    print(f"Final test-set novel 3-gram ratio: {test_novel_ratio:.4f} (closer to 0 is better)")


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 5/5 [00:13<00:00,  2.68s/it]


Final test-set metrics: {'test_loss': 0.2688654661178589, 'test_model_preparation_time': 0.0, 'test_rougeL': 0.38265803295512424, 'test_runtime': 16.3437, 'test_samples_per_second': 1.224, 'test_steps_per_second': 0.306}
Final test-set novel 3-gram ratio: 0.1000 (closer to 0 is better)


## 6. Qualitative check — run maintenance rehearsal on a real chunked article

Feeds the just-trained model into `rehearse_maintenance` (query-agnostic,
`src/pipeline/rehearsal.py`) against chunks from a real article — a
`cnn_dailymail` **test**-split article (disjoint from the train/validation
data used above), so this isn't just re-compressing something the model
already saw — and checks whether compression happens and novel n-gram ratio
stays low. This is a spot-check on individual, human-readable outputs;
§5 above is the actual quantitative test-set result.

In [7]:
if not DATA_READY:
    print("No data — skipping validation.")
elif not API_KEY_SET:
    print("No NVIDIA_NIM_API_KEY (verbatim snapping needs the embedding API) — skipping validation.")
else:
    import yaml
    from datasets import load_dataset

    from src.pipeline.chuncking import paginate_semantic, plain_text_to_paragraphs
    from src.pipeline.embeddings import embed_texts, load_config
    from src.pipeline.rehearsal import rehearse_maintenance

    test_article = load_dataset("cnn_dailymail", "3.0.0", split="test[0:1]")[0]["article"]

    paragraphs = plain_text_to_paragraphs(test_article)
    chunk_cfg = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))
    embed_cfg = load_config("configs/importance_filter.yaml")

    chunks = paginate_semantic(
        paragraphs,
        min_words=chunk_cfg["min_words"],
        max_words=chunk_cfg["max_words"],
        granularity="paragraph",
        config=embed_cfg,
        embed_fn=embed_texts,
    )
    sample_chunks = chunks[:5]
    print(f"Validation chunks: {len(sample_chunks)} (of {len(chunks)} total)")

    rehearsed = rehearse_maintenance(
        sample_chunks, model=model, tokenizer=tokenizer, embed_cfg=embed_cfg, embed_fn=embed_texts
    )

    ngram_ratios = []
    for original, compressed in zip(sample_chunks, rehearsed):
        ratio = novel_ngram_ratio(compressed.text, original.text, n=3)
        ngram_ratios.append(ratio)
        compression = len(compressed.text) / max(1, len(original.text))
        print(f"\n[chunk {original.index}] source {len(original.text)} chars -> "
              f"compressed {len(compressed.text)} chars ({compression:.1%})")
        print(f"  novel 3-gram ratio: {ratio:.4f}")
        print(f"  compressed: {compressed.text[:200]}")

    print(f"\nAverage novel 3-gram ratio: {sum(ngram_ratios) / len(ngram_ratios):.4f} (closer to 0 is better)")


Validation chunks: 3 (of 3 total)

[chunk 0] source 1040 chars -> compressed 312 chars (30.0%)
  novel 3-gram ratio: 0.0000
  compressed: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territor

[chunk 1] source 997 chars -> compressed 320 chars (32.1%)
  novel 3-gram ratio: 0.0000
  compressed: "As the Rome Statute today enters into force for the State of Palestine, Palestine acquires all the rights as well as responsibilities that come with being a State Party to the Statute. These are subs

[chunk 2] source 1573 chars -> compressed 355 chars (22.6%)
  novel 3-gram ratio: 0.0000
  compressed: "What's objectionable is the attempts to undermine international justice, not Palestine's decision to join a treaty to which over 100 countries around the world are members." In January, when the prel

Average novel 3-gram ratio: 0.0000 (closer to 0 is 

## Summary

- This pilot: `PILOT_TRAIN_SIZE`/`PILOT_VAL_SIZE`/`PILOT_TEST_SIZE` = 200/20/20
  of `cnn_dailymail`, `t5-small`, 1 epoch (50 steps). Purely a code-path
  check, not a quality benchmark — but the numbers came back genuinely
  healthy: train loss 0.91 → 0.40, validation `eval_rougeL` 0.218, and (the
  number that actually matters) **held-out test_rougeL 0.379** — better
  than validation, on data the model never saw in any form during training
  or checkpoint selection. Test-set novel 3-gram ratio: **0.100**, consistent
  with validation's 0.106. The qualitative check (§6, a different held-out
  `cnn_dailymail` test article) showed **novel 3-gram ratio = 0.0000 on all
  3 chunks** — every generated compression was fully verbatim-snapped,
  exactly the property A is supposed to have.
- Training was also much faster than the old pko-t5-small pilot on the same
  hardware (MPS): ~115s total here vs. ~25 min before — expected, since
  English `t5-small` sequences are shorter (see 05's percentile note:
  ~1.43 tokens/word vs. pko-t5's ~2.79) and the model itself is smaller
  (60.5M params vs. pko-t5-small's 95.6M).
- Next: (1) scale up `PILOT_TRAIN_SIZE`/`PILOT_TEST_SIZE` (measure step
  timing again first — this pilot's speed doesn't guarantee linear scaling),
  (2) swap backbone to `t5-base`, (3) if novel n-gram ratio regresses at
  scale, debug whether verbatim snapping (`_snap_to_verbatim`) is actually
  snapping correctly on individual samples, (4) record results in Obsidian
  `11_되뇌기 구현 가이드.md`.